In [ ]:
# text preprocessing
import spacy
import pandas as pd

# used spaCy due to lemmatization issues with nltk , only en_core_web_sm is needed
# https://spacy.io/usage/linguistic-features
# https://spacy.io/models/en

nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser']) 
df = pd.read_csv("MASTER_SAMPLED.csv")


def preprocess(text):
    if not isinstance(text, str):     # ignore punctuation
      return ""
    
    doc = nlp(text)                   # tokenize, lemmatize, remove stop words,
    clean_tokens = [                  # lowercase, remove short words (< 2)
          token.lemma_.lower()
          for token in doc 
          if not token.is_stop and token.is_alpha and len(token.text) > 2
      ]

    return " ".join(clean_tokens)

df['clean_t'] = df['full_text'].apply(preprocess)
p_df = df[['title','region','domain','date','clean_t']]
p_df.to_csv("master_preprocess.csv", index=False)

In [ ]:
# coherence and perplexity tests
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

df = pd.read_csv('master_preprocess.csv')

processed_doc = [str(text).split() for text in df['clean_t']]

# indiv bag-of-words corpus building
# create a dict of every unique word
id2word = corpora.Dictionary(processed_doc)

corpus = [id2word.doc2bow(doc) for doc in processed_doc]        # convert articles into bow (wordID, count)

k_values = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]           # testing k from 4 to 15 topics to find 
coherence_scores = []                                           # the sharpest decrease/peak
perplexity_scores = []

for k in k_values:
    lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=k, random_state=42, passes=10)
    
    # logged perplexity (lower is better)
    perplexity_scores.append(lda_model.log_perplexity(corpus))
    
    # coherence (higher is better)
    coherence_model = CoherenceModel(model=lda_model, texts=processed_doc, dictionary=id2word, coherence='c_v')
    coherence_scores.append(coherence_model.get_coherence())
    print(f"tested k={k}: coherence={coherence_scores[-1]:.4f}, perplexity={perplexity_scores[-1]:.4f}")

# https://radimrehurek.com/gensim/models/ldamodel.html

In [ ]:
# LDA model for k=5 topics

processed_doc = [str(text).split() for text in df['clean_t']] 
id2word = corpora.Dictionary(processed_doc)
corpus = [id2word.doc2bow(doc) for doc in processed_doc]

optimal_k = 5
lda_final = LdaModel(
    corpus=corpus, 
    id2word=id2word, 
    num_topics=optimal_k, # k=5
    random_state=42, 
    passes=20, 
    alpha='auto'
)

print(f"top 10 words for {optimal_k} amount of topics")
for idx, topic in lda_final.print_topics(-1):
    print(f"topic {idx}: {topic}\n")

# add a dominant_topic column to df
def get_dominant_topic(bow):
    topic_probs = lda_final.get_document_topics(bow)
    # topic with the highest probability
    dominant = sorted(topic_probs, key=lambda x: x[1], reverse=True)[0][0]
    return dominant

In [ ]:
# find top frequency words in whole dataset with collections python module
# https://docs.python.org/3/library/collections.html#collections.Counter

from collections import Counter

all_words = [word for text in df['clean_t'].dropna() for word in str(text).split()]

word_freq = Counter(all_words)

print("80 most frequent words:")
for word, count in word_freq.most_common(80):
    print(f"{word}: {count}")

In [ ]:
# use frequent words list to make a global noise set and reprocess LDA model

global_noise = set([
    'say','want','greenland','trump','denmark','president','danish','united','states', # 'united' may refer to several diff things
    'country','donald','world','year','tell','greenlandic','include','go','come','add','time','think','need','new','people','week','way','call','like','post'
]) # baseline & functional noise , verbs/nouns , adverbs

df = pd.read_csv('master_preprocess.csv')

def final_clean(text_list):
    return [w for w in text_list if w not in global_noise]

processed_doc = [str(text).split() for text in df['clean_t']] 
sharper_docs = [final_clean(doc) for doc in processed_doc]
id2word_sharp = corpora.Dictionary(processed_doc)

corpus_sharp = [id2word_sharp.doc2bow(doc) for doc in processed_doc]

lda_sharp = LdaModel(
    corpus=corpus_sharp, 
    id2word=id2word_sharp, 
    num_topics=5, 
    random_state=42, 
    passes=15,  # passes for better convergence
)

for idx, topic in lda_sharp.print_topics(-1):
    print(f"topic {idx}: {topic}\n")

In [ ]:
# now region-specific LDA with and without global noise (manually adjust)

def final_clean(text_list):
    return [w for w in text_list if w not in global_noise and len(w) > 2]

all_region_results = []
df = pd.read_csv('master_preprocess.csv')
regions = df['region'].unique()

for reg in regions:
    print(f"processing region {reg}...")
    subset = df[df['region'] == reg]

    processed_doc = [str(text).split() for text in subset['clean_t']] 
    id2word = corpora.Dictionary(processed_doc)
    corpus = [id2word.doc2bow(doc) for doc in processed_doc]
    
    lda = LdaModel(
        corpus=corpus, 
        id2word=id2word, 
        num_topics=5, 
        random_state=42, 
        passes=30, 
        alpha='auto'
    )
    
    for idx, topic in lda.print_topics(-1):
        all_region_results.append({
            'region': reg,
            'topic_id': idx,
            'top_words': topic
        })

results_df = pd.DataFrame(all_region_results)
results_df.to_csv('ldaresults_withgnoise.csv', index=False)

In [ ]:
# prepare LDA matches
import numpy as np
import re

df_lda = pd.read_csv("ldaresults_withgnoise.csv")

general_raw = [
    ("general", 0, '0.0.022*"greenland" + 0.014*"trump" + 0.013*"say" + 0.010*"party" + 0.010*"independence" + 0.009*"election" + 0.008*"denmark" + 0.006*"greenlandic" + 0.006*"people" + 0.006*"nuuk"'),
    ("general", 1, '0.019*"greenland" + 0.007*"company" + 0.007*"year" + 0.005*"say" + 0.005*"project" + 0.004*"market" + 0.004*"new" + 0.004*"ice" + 0.004*"mining" + 0.004*"earth"'),
   ("general", 2, '0.032*"trump" + 0.019*"greenland" + 0.015*"say" + 0.012*"president" + 0.008*"tariff" + 0.007*"european" + 0.006*"nato" + 0.006*"world" + 0.006*"denmark" + 0.005*"united"'),
    ("general", 3, '0.047*"greenland" + 0.020*"trump" + 0.016*"say" + 0.016*"denmark" + 0.009*"president" + 0.009*"danish" + 0.009*"arctic" + 0.008*"island" + 0.008*"security" + 0.007*"visit"'),
    ("general", 4, '0.040*"greenland" + 0.024*"say" + 0.020*"denmark" + 0.018*"trump" + 0.013*"nato" + 0.011*"danish" + 0.010*"arctic" + 0.010*"military" + 0.010*"minister" + 0.008*"president"')
]

def parse_lda_string(lda_str):
    if pd.isna(lda_str): return {}
    matches = re.findall(r'(\d\.\d+)\*"([^"]+)"', str(lda_str))
    return {word: float(weight) for weight, word in matches}

parse_data = []
vocab = set()

for idx, row in df_lda.iterrows():
    w_map = parse_lda_string(row['top_words'])
    parse_data.append({'region': row['region'], 'topic_id': row['topic_id'], 'weights': w_map})
    vocab.update(w_map.keys())

for reg, tid, words_str in general_raw:
    w_map = parse_lda_string(words_str)
    parse_data.append({'region': reg, 'topic_id': tid, 'weights': w_map})
    vocab.update(w_map.keys())

vocab_list = sorted(list(vocab))
matrix = np.zeros((len(parse_data), len(vocab_list))) # feature matrix

for i, item in enumerate(parse_data):
    for word, weight in item['weights'].items():
        matrix[i, vocab_list.index(word)] = weight

In [ ]:
# compute LDA matches
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load('en_core_web_md')

anchors = {
    "geopolitical sovereignity & security": "military nato minister arctic security island territory prime ally foreign",
    "natural resources & global interests": "company project mineral mining market rare molybdenum development million earth",
    "geographic & strategic positioning": "arctic island ice base china military nuuk north mineral resource",
    "economic pressure & relationships": "european tariff nato europe force deal threat davos ally economic",
    "diplomatic engagement": "visit vance security base tariff vice minister government territory usha"
}

anchor_vecs = {k: nlp(v).vector for k, v in anchors.items()}

def clean_lda(lda_str):
    # extracts  words from string
    return " ".join(re.findall(r'"([^"]*)"', str(lda_str)))

df = pd.read_csv('ldaresults.csv')
final_output = []

for _, row in df.iterrows():
    cleaned_words = clean_lda(row['top_words'])
    topic_vec = nlp(cleaned_words).vector.reshape(1, -1)
    
    scores = {}
    for label, a_vec in anchor_vecs.items():
        score = cosine_similarity(topic_vec, a_vec.reshape(1, -1))[0][0] # find best match
        scores[label] = score
    
    best_label = max(scores, key=scores.get) # top performance
    max_score = round(scores[best_label], 4)
    
    final_output.append({
        "region": row['region'],
        "topic_id": row['topic_id'],
        "best_match": best_label,
        "score": max_score
    })

output_df = pd.DataFrame(final_output)
output_df.to_csv('ldamatches.csv', index=False)

print(output_df.head())

In [ ]:
# PCA and bi-plots with and without global noise
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA

plt.style.use('seaborn-v0_8-whitegrid') 

df = pd.read_csv("lda/ldaresults_withgnoise.csv")  

def parse_top_words(top_words_str, n_words=10):
    pairs = re.findall(r'([\d.]+)\*"(\w+)"', top_words_str)
    vec = {}
    for weight, word in pairs[:n_words]:
        vec[word] = float(weight)
    return vec

# build vocab
all_entries = []
for _, row in df.iterrows():
    wvec = parse_top_words(row["top_words"])
    all_entries.append({
        **wvec,
        "region": row["region"],
        "topic_id": row["topic_id"],
    })

labels  = [f"{e['region'][:3].upper()} {e['topic_id']}" for e in all_entries]
regions = [e["region"] for e in all_entries]

topic_df = pd.DataFrame(all_entries).fillna(0)

# feature matrix
feature_cols = [c for c in topic_df.columns if c not in ["region", "topic_id"]] # each row = one topic (regional or general)
X_scaled = topic_df[feature_cols].values # columns as word weights

# DON'T standardize - makes every word have an equal weight which is not what we want

pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_scaled)

print("explained variance")
print("-" * 60)
for i, (var, cum) in enumerate(zip(
        pca.explained_variance_ratio_,
        np.cumsum(pca.explained_variance_ratio_))):
    print(f"  PC{i+1}: {var*100:.1f}%  (cumulative: {cum*100:.1f}%)")

print()

print("PC loadings: top words driving each component")
print("-" * 60)
for pc_idx in range(2):  # just PC1 and PC2 for thesis
    loadings = pca.components_[pc_idx]
    sorted_idx = np.argsort(np.abs(loadings))[::-1]
    print(f"\nPC{pc_idx + 1} — top words (by absolute loading):")
    print(f"  {'word':<20} {'loading':>10}  {'direction'}")
    print(f"  {'-'*20} {'-'*10}  {'-'*20}")
    for idx in sorted_idx[:10]:
        word = feature_cols[idx]
        load = loadings[idx]
        direction = "positive pole" if load > 0 else "negative pole"
        print(f"  {word:<20} {load:>10.4f}  {direction}")

print()

print("topic positions on PC1 & PC2")
results = topic_df[["region", "topic_id"]].copy()
results["PC1"] = X_pca[:, 0]
results["PC2"] = X_pca[:, 1]

print(results.to_string(index=False))
print()

print("mean PC1 / PC2 per region (centroid)")
for region in results["region"].unique():
    sub = results[results["region"] == region]
    print(f"  {region:<25}  PC1={sub['PC1'].mean():.4f}  PC2={sub['PC2'].mean():.4f}")

print()

print("interpretation: PC1 poles")
pc1_loads = pca.components_[0]
pos_words = [(feature_cols[i], pc1_loads[i]) for i in np.argsort(pc1_loads)[-5:]]
neg_words = [(feature_cols[i], pc1_loads[i]) for i in np.argsort(pc1_loads)[:5]]
print("PC1 positive pole (right side of plot):", [w for w, _ in reversed(pos_words)])
print("PC1 negative pole (left side of plot): ", [w for w, _ in reversed(neg_words)])

pc2_loads = pca.components_[1]
pos_words2 = [(feature_cols[i], pc2_loads[i]) for i in np.argsort(pc2_loads)[-5:]]
neg_words2 = [(feature_cols[i], pc2_loads[i]) for i in np.argsort(pc2_loads)[:5]]
print("PC2 positive pole (top of plot):        ", [w for w, _ in reversed(pos_words2)])
print("PC2 negative pole (bottom of plot):     ", [w for w, _ in reversed(neg_words2)])

palette = {'US': 'indianred', 
           'affected europe': 'darkgoldenrod', 
           'greenland & denmark': 'darkseagreen', 
           'other': 'lightslategrey'}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# left: scatter
ax = axes[0]
for _, row in results.iterrows():
    color = palette.get(row["region"], "#999")
    ax.scatter(row["PC1"], row["PC2"], color=color, s=120, zorder=3)
    ax.annotate(f"{row['region'][:3]} {row['topic_id']}",
                (row["PC1"], row["PC2"]),
                fontsize=7.5, xytext=(4, 4), textcoords="offset points")

# centroids
for region, color in palette.items():
    sub = results[results["region"] == region]
    cx, cy = sub["PC1"].mean(), sub["PC2"].mean()
    ax.scatter(cx, cy, color=color, s=300, marker="*", zorder=5,
               edgecolors="white", linewidths=0.8)

legend_handles = [mpatches.Patch(color=c, label=r) for r, c in palette.items()]
legend_handles.append(plt.scatter([], [], marker="*", color="gray", s=200, label="centroid"))
ax.set_facecolor("linen")
ax.legend(handles=legend_handles, fontsize=8, loc="upper left")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=11)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=11)
# ax.set_title("PCA of LDA topic distributions", fontsize=12)
ax.grid(True, alpha=0.3, linewidth=0.5)
ax.axhline(0, color="gray", linewidth=0.5, alpha=0.5)
ax.axvline(0, color="gray", linewidth=0.5, alpha=0.5)

# right: biplot with arrows
ax2 = axes[1]
for _, row in results.iterrows():
    color = palette.get(row["region"], "#999")
    ax2.scatter(row["PC1"], row["PC2"], color=color, s=80, alpha=0.6, zorder=3)

# arrows for top 8 words
combined_load = np.sqrt(pca.components_[0]**2 + pca.components_[1]**2)
top_word_idx = np.argsort(combined_load)[-8:]
scale = 0.5 * max(np.abs(X_pca[:, :2]).max(axis=0)) # scale arrows

for idx in top_word_idx:
    lx = pca.components_[0][idx] * scale
    ly = pca.components_[1][idx] * scale
    ax2.annotate("", xy=(lx, ly), xytext=(0, 0),
                 arrowprops=dict(arrowstyle="->", color="darkmagenta", lw=1.5))
    ax2.text(lx * 1.15, ly * 1.15, feature_cols[idx],
             fontsize=8, color="darkmagenta", ha="center")

ax2.set_facecolor("linen")
ax2.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=11)
ax2.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=11)
# ax2.set_title("biplot: word loadings", fontsize=12)
ax2.grid(True, alpha=0.3, linewidth=0.5)
ax2.axhline(0, color="gray", linewidth=0.5, alpha=0.5)
ax2.axvline(0, color="gray", linewidth=0.5, alpha=0.5)
ax2.legend(handles=legend_handles[:4], fontsize=8, loc="upper left")

plt.tight_layout()
plt.savefig("pca_analysis_unscaled_gnoise.png", dpi=180, bbox_inches="tight")
plt.close()

# scree
fig2, ax3 = plt.subplots(figsize=(6, 4))
pc_labels = [f"PC{i+1}" for i in range(5)]
ax3.bar(pc_labels, pca.explained_variance_ratio_ * 100, color="darkgoldenrod", alpha=0.8)
ax3.plot(pc_labels, np.cumsum(pca.explained_variance_ratio_) * 100,
         "o-", color="firebrick", label="cumulative")
ax3.set_facecolor("linen")
ax3.set_ylabel("explained variance (%)")
# ax3.set_title(" scree plot: PCA on LDA topic distributions")
ax3.legend()
ax3.grid(True, alpha=0.3, linewidth=0.5)
plt.tight_layout()
plt.savefig("pca_scree_unscaled_gnoise.png", dpi=180, bbox_inches="tight")
plt.close()

# https://plotivy.app/blog/how-to-create-pca-biplot-in-python

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import re

df = pd.read_csv("lda/ldaresults.csv")

REGION_COLORS = {
    "US":                   "indianred",
    "affected europe":      "darkgoldenrod",
    "greenland & denmark":  "darkseagreen",
    "other":                "lightslategrey",
}

def parse_topic_weights(top_words_str):
    pairs = re.findall(r'([\d.]+)\*"(\w+)"', top_words_str) # parse LDA top words to dictionary
    return {word: float(weight) for weight, word in pairs}

def make_wordcloud(region, topic_id, top_words_str, color, ax):
    weights = parse_topic_weights(top_words_str)
    wc = WordCloud(
        width=500, height=300,
        background_color="linen",
        colormap="RdYlGn",
        max_words=30,
        prefer_horizontal=0.9,
        random_state=42
    ).generate_from_frequencies(weights)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(f"{region} — topic {topic_id}", fontsize=10,
                 color=color, fontweight="bold", pad=6)

# one topic per region
def plot_one_topic_per_region(topic_id):
    regions = ["US", "affected europe", "greenland & denmark", "other"]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.patch.set_facecolor("white")
    for ax, region in zip(axes, regions):
        row = df[(df["region"] == region) & (df["topic_id"] == topic_id)]
        if len(row) == 0:
            ax.axis("off")
            continue
        color = REGION_COLORS.get(region, "gray")
        make_wordcloud(region, topic_id, row.iloc[0]["top_words"], color, ax)
    plt.tight_layout()
    plt.savefig(f"wordcloud_topic{topic_id}_comparison.png", dpi=180,
                bbox_inches="tight", facecolor="white")
    plt.close()

plot_one_topic_per_region(3)
plot_one_topic_per_region(4)

# https://medium.com/@kenny_39527/an-overview-of-lda-and-wordcloud-analysis-of-python-tutorials-on-w3schools-13763d25fa3a